# High-rise 01 - Inlet profile validation

Validate the atmospheric boundary layer: detect the vertical profiles in the
probe cloud, plot mean velocity / turbulence intensity / spectrum into `debug/`,
and extract the simulation mean velocity at the reference height (`u_ref`). That
`u_ref` feeds the Cp non-dimensionalisation in notebook 02.

Defaults run on the `pitot_inlet` fixture. Point at a real case by setting the
`CFDMOD_HR_INFLOW_HIST` / `CFDMOD_HR_INFLOW_POINTS` / `CFDMOD_HR_REF_HEIGHT`
environment variables (or editing the config cell).

In [ ]:
import os
import pathlib

import numpy as np  # noqa: F401  (used across later cells)

from cfdmod import high_rise as hr
from cfdmod.high_rise import plotting


def _find_repo(start: pathlib.Path) -> pathlib.Path:
    p = start.resolve()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    return start.resolve()


REPO = _find_repo(pathlib.Path.cwd())

plotting.apply_style()

FIX = REPO / "fixtures" / "tests"
OUTPUT_BASE = pathlib.Path(
    os.environ.get("CFDMOD_HR_OUTPUT_BASE", REPO / "examples" / "high_rise" / "_run")
)
VERSION = os.environ.get("CFDMOD_HR_VERSION", "example")
print("REPO:", REPO)
print("OUTPUT_BASE:", OUTPUT_BASE, "| VERSION:", VERSION)

In [ ]:
from cfdmod.inflow import InflowData, NormalizationParameters

# --- config -------------------------------------------------------------
HIST = pathlib.Path(
    os.environ.get(
        "CFDMOD_HR_INFLOW_HIST", FIX / "inflow" / "pitot_inlet" / "hist_series.csv"
    )
)
POINTS = pathlib.Path(
    os.environ.get("CFDMOD_HR_INFLOW_POINTS", FIX / "inflow" / "pitot_inlet" / "points.csv")
)
REFERENCE_HEIGHT = float(os.environ.get("CFDMOD_HR_REF_HEIGHT", "2.0"))
COMPONENT = os.environ.get("CFDMOD_HR_COMPONENT", "ux")

dbg = hr.DebugWriter(OUTPUT_BASE, stage="inflow", version=VERSION)
inflow = InflowData.from_files(HIST, POINTS)
profiles = hr.detect_profiles(inflow, min_points=3)
print(f"detected {len(profiles)} vertical profile(s)")
for p in profiles:
    print(f"  {p.name}: {len(p.point_idx)} points, z in [{p.z.min():.2f}, {p.z.max():.2f}]")

In [ ]:
# --- per-profile figures + reference velocity ---------------------------
u_ref_by_profile = {}
for prof in profiles:
    u_ref = hr.reference_velocity(prof, inflow, REFERENCE_HEIGHT, component=COMPONENT)
    u_ref_by_profile[prof.name] = u_ref
    L = hr.inflow_report.integral_length_scale(
        inflow, prof.nearest_index(REFERENCE_HEIGHT), u_ref, component=COMPONENT
    )
    print(f"{prof.name}: u_ref(z={REFERENCE_HEIGHT:g}) = {u_ref:.4f} m/s | L = {L:.4g} m")

    norm = NormalizationParameters(
        reference_velocity=max(u_ref, 1e-9), characteristic_length=1.0
    )
    for name, fig in {
        "mean_velocity": hr.inflow_report.plot_mean_velocity(prof, inflow, component=COMPONENT),
        "turbulence_intensity": hr.inflow_report.plot_turbulence_intensity(
            prof, inflow, component=COMPONENT
        ),
        "spectrum": hr.inflow_report.plot_spectrum(
            prof, inflow, REFERENCE_HEIGHT, norm, component=COMPONENT
        ),
    }.items():
        dbg.savefig(fig, f"{prof.name}/{name}.png")
        plotting.close(fig)
print("figures written under", dbg.debug_dir)

In [ ]:
import json

# --- persist u_ref for the Cp step (the 'update config' step) -----------
# The richest profile is the representative inlet column.
primary = profiles[0].name if profiles else None
u_ref = u_ref_by_profile.get(primary)
out = dbg.deliverable_path("reference_velocity.json")
out.write_text(
    json.dumps(
        {"profile": primary, "reference_height": REFERENCE_HEIGHT, "u_ref": u_ref},
        indent=2,
    )
)
print(f"u_ref = {u_ref} m/s -> {out}")
# In notebook 02: case = case.with_reference_velocity(u_ref)